In [ ]:
# Cell 1: Import necessary libraries and environment setup
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Wav2Vec2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, accuracy_score

# Configure device and check GPU specifications
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("-" * 50)
print(f"🚀 Execution Device: {device}")

if torch.cuda.is_available():
    # Let's flex the hardware for your teammates!
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3) # Convert to GB
    print(f"🔥 Hardware Model: {gpu_name}")
    print(f"📟 Dedicated VRAM: {gpu_memory:.2f} GB")
    print(f"✨ CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️ Warning: Running on CPU. Performance might be limited.")
print("-" * 50)

# Global setting: Optimization for large-scale audio processing
# Note: We are using librosa for loading to bypass torchaudio/torchcodec version conflicts
import librosa 
print("✅ Environment initialized. Ready for FinQuest 2026.")

In [ ]:
# Cell 2: Load Master Protocol and Perform Dataset Splitting (Train/Dev/Test)
# We partition the data using an 80/10/10 ratio for Training, Development, and Testing.
protocol_path = "./protocol/master_protocol.csv"
df = pd.read_csv(protocol_path)

# First split: Separate 80% for Training and 20% for the remaining pool (Temp)
# Note: 'stratify' ensures the Real/Fake ratio remains consistent across all sets.
train_df, temp_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['label'], 
    random_state=42
)

# Second split: Evenly divide the remaining 20% into Dev and Test sets (10% each of total)
dev_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['label'], 
    random_state=42
)

print("-" * 60)
print(f"📊 Dataset Statistics:")
print(f"Total Samples: {len(df)}")
print(f"Training Set : {len(train_df)} samples (80%)")
print(f"Validation Set: {len(dev_df)} samples (10%)")
print(f"Testing Set   : {len(test_df)} samples (10%)")
print("-" * 60)

# Verification of class balance for the team
print("✅ Stratification Check (Real vs Fake ratio):")
print(f"Train Label 1 Ratio: {train_df['label'].mean():.2%}")
print(f"Dev   Label 1 Ratio: {dev_df['label'].mean():.2%}")
print(f"Test  Label 1 Ratio: {test_df['label'].mean():.2%}")
print("-" * 60)

In [ ]:
# Cell 3: Robust Audio Dataset Definition
import librosa
import torch
import torchaudio

class FinQuestDataset(Dataset):
    """
    Custom Dataset for Deepfake Detection.
    Uses librosa for high-stability audio loading and automatic resampling.
    """
    def __init__(self, dataframe, processor, max_length=64000):
        self.df = dataframe
        self.processor = processor
        self.max_length = max_length  # Default 64000 samples = 4 seconds @ 16kHz

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row['path']
        label = row['label']

        try:
            # Step 1: Load audio using librosa for maximum compatibility
            # Forced resampling to 16,000Hz and mono conversion
            waveform, _ = librosa.load(path, sr=16000, mono=True)
            waveform = torch.from_numpy(waveform)
            
            # Step 2: Ensure uniform sequence length (Truncate or Zero-pad)
            if len(waveform) > self.max_length:
                waveform = waveform[:self.max_length]
            else:
                padding_needed = self.max_length - len(waveform)
                waveform = torch.nn.functional.pad(waveform, (0, padding_needed))

            # Step 3: Feature Extraction via Wav2Vec2 Processor
            inputs = self.processor(waveform, sampling_rate=16000, return_tensors="pt")
            
            return {
                "input_values": inputs.input_values.squeeze(0),
                "label": torch.tensor(label, dtype=torch.long)
            }
        except Exception as e:
            # Recursive Fallback: If a file is corrupted, skip to the next valid sample
            # This prevents the training loop from crashing during large-scale runs
            # print(f"Warning: Skipping corrupted file {path} - {e}")
            return self.__getitem__((idx + 1) % len(self.df))

print("✅ FinQuestDataset class defined with robust error handling and auto-resampling.")

In [ ]:
# Cell 4: Model Architecture Definition (Wav2Vec2 + Attentive Pooling + MLP)

class AttentivePooling(torch.nn.Module):
    """
    Self-Attentive Pooling layer to aggregate frame-level features into a 
    single utterance-level representation by focusing on relevant temporal segments.
    """
    def __init__(self, hidden_size):
        super().__init__()
        # Attention mechanism to compute importance scores for each frame
        self.attention = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, 128),
            torch.nn.Tanh(),
            torch.nn.Linear(128, 1)
        )

    def forward(self, x):
        # Input shape x: (batch, seq_len, hidden_size)
        # Compute softmax-normalized attention weights across the time dimension
        attn_weights = torch.softmax(self.attention(x), dim=1)
        # Weighted summation of hidden states
        pooled = torch.sum(attn_weights * x, dim=1)
        return pooled

# Step 1: Load Pre-trained Wav2Vec 2.0 Base Engine
# This model provides robust acoustic feature extraction from raw audio.
processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")
backbone = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)

# Step 2: Initialize Classification Head
# 768 is the hidden size for the wav2vec2-base transformer layers.
pooling = AttentivePooling(768).to(device)

# Multi-Layer Perceptron (MLP) for final binary classification (Real vs Fake)
classifier = torch.nn.Sequential(
    torch.nn.Linear(768, 256),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),  # Regularization to prevent overfitting
    torch.nn.Linear(256, 2)
).to(device)

print(f"✅ Model architecture initialized on {device}.")
print(f"🔹 Backbone: Wav2Vec2-Base (Frozen/Fine-tuned)")
print(f"🔹 Aggregation: Self-Attentive Pooling")
print(f"🔹 Output: Binary Logits [Fake, Real]")

In [ ]:
# Cell 5: Training Hyperparameters & Optimizer Configuration (LLRD Version)

# Initialize Training and Validation Datasets
train_dataset = FinQuestDataset(train_df, processor)
dev_dataset   = FinQuestDataset(dev_df, processor)

# DataLoaders: batch_size=4 is used for RTX 5080 VRAM efficiency. 
# Increasing batch_size to 8 may provide more stable gradient estimates if VRAM allows.
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4)
dev_loader   = DataLoader(dev_dataset, batch_size=4, num_workers=4)

# Layer-wise Learning Rate Decay (LLRD) Strategy:
# We apply differential learning rates to preserve pre-trained knowledge while 
# allowing the classification head to adapt quickly to deepfake-specific patterns.
optimizer = torch.optim.AdamW([
    # 1. Low-level CNN Feature Extractor: 1e-6 (Keep low to protect basic acoustic filters)
    {"params": backbone.feature_extractor.parameters(), "lr": 1e-6}, 
    
    # 2. Mid-level Transformer Encoder: 5e-6 (Fine-tune contextual representations)
    {"params": backbone.encoder.parameters(), "lr": 5e-6},
    
    # 3. High-level Pooling & Classifier Heads: 5e-5 (Focus on learning new spoofing artifacts)
    {"params": pooling.parameters(), "lr": 5e-5},
    {"params": classifier.parameters(), "lr": 5e-5},
], weight_decay=0.01)

# Standard Loss Function for Binary Classification (Real vs Fake)
criterion = torch.nn.CrossEntropyLoss()

print("-" * 60)
print("🚀 Training Configuration Summary:")
print(f"🔹 Optimizer    : AdamW with LLRD")
print(f"🔹 Batch Size   : {train_loader.batch_size}")
print(f"🔹 CNN LR       : 1e-06")
print(f"🔹 Encoder LR   : 5e-06")
print(f"🔹 Head LR      : 5e-05")
print(f"🔹 Weight Decay : 0.01")
print("-" * 60)
print("✅ Optimization engine ready for fine-tuning.")

In [ ]:
# Cell 6: Training & Validation Loop (with Early Stopping and Model Versioning)
import os

# Ensure the model storage directory exists
MODEL_DIR = "./model"
os.makedirs(MODEL_DIR, exist_ok=True)

def evaluate(model_base, pool_layer, clf_layer, loader):
    """
    Performs validation inference and calculates key performance metrics.
    Metrics: Accuracy (ACC) and Equal Error Rate (EER).
    """
    model_base.eval()
    pool_layer.eval()
    clf_layer.eval()
    all_scores, all_labels = [], []
    
    with torch.no_grad():
        for batch in loader:
            inputs = batch["input_values"].to(device)
            labels = batch["label"].to(device)
            
            # Forward pass through the full pipeline
            outputs = model_base(inputs).last_hidden_state
            pooled = pool_layer(outputs)
            logits = clf_layer(pooled)
            
            # Extract probability for Class 1 (Real)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_scores.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            
    # Compute Equal Error Rate (EER) - The industry standard for spoofing detection
    fpr, tpr, _ = roc_curve(all_labels, all_scores, pos_label=1)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fnr - fpr))]
    
    # Compute standard Accuracy at 0.5 threshold
    acc = accuracy_score(all_labels, np.array(all_scores) > 0.5)
    return acc, eer

# --- Training Configuration & Early Stopping ---
best_eer = 1.0     # Initial threshold for model saving
patience = 3       # Stop training if EER doesn't improve for 3 consecutive epochs
counter = 0        # Early stopping counter
max_epochs = 10    # Maximum number of training iterations

print("🏁 Starting training loop for FinQuest 2026...")

for epoch in range(max_epochs):
    backbone.train()
    pooling.train()
    classifier.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    for batch in pbar:
        optimizer.zero_grad()
        
        inputs = batch["input_values"].to(device)
        labels = batch["label"].to(device)
        
        # Forward pass
        outputs = backbone(inputs).last_hidden_state
        pooled = pooling(outputs)
        logits = classifier(pooled)
        
        # Loss computation and Backpropagation
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.6f}")

    # --- Validation Phase ---
    dev_acc, dev_eer = evaluate(backbone, pooling, classifier, dev_loader)
    avg_loss = total_loss / len(train_loader)
    
    print(f"\n📊 Epoch {epoch} Summary: Loss={avg_loss:.4f}, Dev Acc={dev_acc:.4f}, Dev EER={dev_eer:.4f}")
    
    # --- Model Saving & Early Stopping Logic ---
    if dev_eer < best_eer:
        best_eer = dev_eer
        counter = 0  # Reset counter on performance improvement
        save_path = os.path.join(MODEL_DIR, "best_deepfake_model.pth")
        
        torch.save({
            'backbone': backbone.state_dict(),
            'pooling': pooling.state_dict(),
            'classifier': classifier.state_dict()
        }, save_path)
        print(f"🥇 New Best Model! Performance improved. Saved to: {save_path}")
    else:
        counter += 1
        print(f"⏱ No improvement. Early Stopping counter: {counter}/{patience}")
    
    if counter >= patience:
        print(f"🛑 Early Stopping triggered. Training halted. Best EER: {best_eer:.4f}")
        break

print("\n✨ Training session complete.")

In [ ]:
# Cell 7: Full Test Evaluation and Model Source Attribution Report
import librosa
from sklearn.metrics import roc_auc_score, confusion_matrix

# 1. Load the optimal weights from the model directory
print("🔄 Loading best model weights for final evaluation...")
checkpoint = torch.load("model/best_deepfake_model.pth")
backbone.load_state_dict(checkpoint['backbone'])
pooling.load_state_dict(checkpoint['pooling'])
classifier.load_state_dict(checkpoint['classifier'])

backbone.eval()
pooling.eval()
classifier.eval()

all_scores = []
all_labels = []
source_results = {} # Metrics storage for cross-algorithm traceability

print(f"🔍 Running full inference on {len(test_df)} unseen test samples...")

with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference Phase"):
        try:
            # Step 1: Pre-processing
            wav, _ = librosa.load(row['path'], sr=16000, mono=True)
            inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_values.to(device)
            
            # Step 2: Forward pass through the inference pipeline
            out = backbone(inputs).last_hidden_state
            pooled = pooling(out)
            logits = classifier(pooled)
            
            # Compute probability for Class 1 (Real)
            prob = torch.softmax(logits, dim=1)[0, 1].item()
            
            label = row['label']
            source = row['model_name'] # Source algorithm (e.g., MelGAN, WaveGAN)
            
            all_scores.append(prob)
            all_labels.append(label)
            
            # Step 3: Attribution tracking
            if source not in source_results:
                source_results[source] = {"total": 0, "correct": 0}
            
            pred = 1 if prob > 0.5 else 0
            source_results[source]["total"] += 1
            if pred == label:
                source_results[source]["correct"] += 1
                
        except Exception as e:
            # Skip corrupted samples if any
            continue

# --- Final Metric Computation ---
all_labels = np.array(all_labels)
all_scores = np.array(all_scores)

# Accuracy Calculation
final_acc = accuracy_score(all_labels, all_scores > 0.5)

# Equal Error Rate (EER) Calculation
fpr, tpr, thresholds = roc_curve(all_labels, all_scores, pos_label=1)
fnr = 1 - tpr
eer = fpr[np.nanargmin(np.abs(fnr - fpr))]

# Area Under the ROC Curve (AUC)
auc_score = roc_auc_score(all_labels, all_scores)

# --- Final Performance Report ---
print("\n" + "="*60)
print("🚀 FINQUEST 2026: COMPREHENSIVE TEST ANALYSIS REPORT")
print("="*60)
print(f"✨ Overall Accuracy       : {final_acc:.2%}")
print(f"✨ Equal Error Rate (EER) : {eer:.4f}")
print(f"✨ Area Under Curve (AUC) : {auc_score:.4f}")
print("-" * 60)
print(f"{'Synthetic Source':<18} | {'Count':<8} | {'Accuracy':<10}")
print("-" * 60)

for src, res in source_results.items():
    src_acc = res['correct'] / res['total']
    # If the source is 'original', it refers to Real samples
    source_label = "Real Samples" if src == "original" else f"Fake ({src})"
    print(f"{source_label:<18} | {res['total']:<8} | {src_acc:.2%}")

print("-" * 60)

# --- Traceability Sample Display ---
print("\n📋 Random Sample Traceability Check:")
print(f"{'Filename':<18} | {'Pred':<6} | {'Conf.':<8} | {'Actual':<6} | {'Algorithm'}")
print("-" * 60)

# Select a subset for visualization
test_samples = test_df.sample(min(10, len(test_df)))
for _, row in test_samples.iterrows():
    wav, _ = librosa.load(row['path'], sr=16000, mono=True)
    inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_values.to(device)
    with torch.no_grad():
        out = backbone(inputs).last_hidden_state
        logits = classifier(pooling(out))
        prob = torch.softmax(logits, dim=1)[0, 1].item()
    
    pred_str = "REAL" if prob > 0.5 else "FAKE"
    actual_str = "REAL" if row['label'] == 1 else "FAKE"
    fname = os.path.basename(row['path'])[:15] + ".."
    
    print(f"{fname:<18} | {pred_str:<6} | {prob:>7.2%} | {actual_str:<6} | {row['model_name']}")

# Clear memory for subsequent tasks
torch.cuda.empty_cache()
print("\n✅ Final evaluation complete.")